# Darcy equation: exercise 3

Let $\Omega=(0,1)^2$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given 
$k$ the matrix permeability, we want to solve the following problem: find $({q}, p)$ such that
$$
\left\{
\begin{array}{ll}
\begin{array}{l} 
k^{-1} {q} + \nabla p = 0\\
\nabla \cdot {q} = 0
\end{array}
&\text{in } \Omega
\end{array}
\right.
$$
with boundary conditions:
$$ p = 1 \text{ on } \partial_{top} \Omega \qquad p = 0 \text{ on } \partial_{bottom} \Omega \qquad \nu \cdot q = 0 \text{ on } \partial_{left} \Omega \cup \partial_{right} \Omega$$
The matrix permeability is defined in the following way
$$
k(x, y) = 
\left\{
\begin{array}{ll}
k_1 & 0.2 < y < 0.4\\
k_2 & 0.6 < y < 0.8\\
k_0=1 & \text{otherwise}
\end{array}
\right.
$$
with, for example, $k_1 = k_2 = 10^{-2}$. Compare the effective permeability computed analytically or numerically.

This is the guided ("fill in the code") version of `ex3.ipynb` -- work through the cells in order, replacing each `# TODO` with your own implementation. Compare against `ex3.ipynb` once you're done, or if you get stuck.

First we import some of the standard modules.

In [ ]:
import numpy as np
import scipy.sparse as sps

import porepy as pp
import pygeon as pg

We create now the grid, to facilitate the imposition of $k$ we consider a structured grid from PorePy and then convert it into a PyGeoN grid.

In [ ]:
# TODO: create a 2d STRUCTURED grid with pg.unit_grid(dim, 1 / N, as_mdg=False, structured=True)
# (try N = 10 to start), then call sd.compute_geometry()


Let us declare the finite element spaces that we are going to use

In [ ]:
# TODO: declare the RT0 (for q) and PwConstants/P0 (for p) discretization
# objects under a key of your choice, e.g. key = "flow"
#
# TODO: build the degrees-of-freedom array
# dofs = np.array([rt0.ndof(sd), p0.ndof(sd)])


With the following code we set the data, in particular the permeability tensor and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need few steps.

In [ ]:
# TODO: build a heterogeneous (inverse) permeability array over the cells: 1/k1 where
# 0.2 < y < 0.4, 1/k2 where 0.6 < y < 0.8 (use sd.cell_centers[1, :] for the y coordinate),
# and 1 (i.e. k0 = 1) everywhere else. Pack it with pp.SecondOrderTensor and
# pp.initialize_data (see pg.SECOND_ORDER_TENSOR)
#
# TODO: identify the four sides of the domain from sd.face_centers (left/right/bottom/top)
#
# TODO: impose p = 1 on top and p = 0 on bottom as a NATURAL boundary condition for RT0
# (use rt0.assemble_nat_bc with a function returning the pressure value on the boundary),
# and nu.q = 0 (essential) on left/right


Once the data are assigned to the grid, we construct the matrices. In particular, the linear system associated with the equation is given as
$$
\left(
\begin{array}{cc} 
A & -B^\top\\
B & 0
\end{array}
\right)
\left(
\begin{array}{c} 
q\\ 
p
\end{array}
\right)
=\left(
\begin{array}{c} 
p_{\partial}\\ 
0
\end{array}
\right)
$$

In [ ]:
# TODO: assemble the local matrices -- the RT0 mass matrix A (with data), the P0 mass
# matrix, and the divergence matrix B = mass_p0 @ rt0.assemble_diff_matrix(sd)
#
# TODO: assemble the saddle-point matrix spp with scipy.sparse.block_array
# ([[A, -B.T], [B, None]], format="csc")
#
# TODO: assemble the right-hand side rhs (length dofs.sum()), adding the boundary
# term to the q-block (the first dofs[0] entries)


We solve the linear system and extract the two solutions $q$ and $p$.

In [ ]:
# TODO: build a pg.LinearSystem from spp and rhs, flag the essential boundary dofs
# with ls.flag_ess_bc(bc_ess, ...), then solve()
#
# TODO: split the solution vector into q and p, e.g. with
# idx = np.cumsum(dofs[:-1]); q, p = np.split(x, idx)


Since the computed $q$ is one value per facet of the grid, for visualization purposes we project the flux in each cell center as vector. We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
# TODO: project q to cell centers with rt0.eval_at_cell_centers(sd), and evaluate p
# at cell centers with p0.eval_at_cell_centers(sd)
#
# TODO: export cell_p, cell_q and the permeability array with
# pp.Exporter(sd, "sol", folder_name="ex3").write_vtu(...)


Let us compute now the effective permeability, analytically we can use the following expression
$$
k_{\perp}^{eff} = \frac{5}{\frac{3}{k_0} + \frac{1}{k_1} + \frac{1}{k_2}}
$$
while, by considering the Darcy law we can approximate numerically the permeability as
$$
 q = - k \nabla p \quad \Rightarrow \quad q \cdot \nu|_{bottom} = - \tilde{k}_{\perp}^{eff} \frac{p_{top} - p_{bottom}}{\Delta y} 
 \quad \Rightarrow \quad \tilde{k}_{\perp}^{eff} = \frac{q \cdot \nu|_{bottom} \Delta y}{p_{top} -p_{bottom}}
$$
by considering the geometry and boundary conditions of the current problem then we obtain
$$
\tilde{k}_{\perp}^{eff} = q \cdot \nu|_{bottom}.
$$

In [ ]:
# TODO: compute the numerical effective permeability as the sum of q over the
# bottom faces (np.sum(q[bottom]))
#
# TODO: compute the analytical effective permeability with the formula above (k0 = 1)
#
# TODO: compute and print the relative error between the two


In [ ]:
# Consistency check -- once your implementation is correct, this should pass
assert np.isclose(np.linalg.norm(relative_err), 0)